## Model Comparison: *qwen/qwen3-32b* vs *llama-3.1-8b-instant*

In [1]:
import os
import sys
sys.path.append('..')

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from src.hybrid_rag_pipeline import get_retriever
from src.context_builder import build_context
from src.prompts import prompt

In [2]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv('GROQ_API_KEY'))
models = client.models.list()
for m in models.data:
    print(m.id)

groq/compound
openai/gpt-oss-safeguard-20b
allam-2-7b
qwen/qwen3-32b
canopylabs/orpheus-arabic-saudi
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound-mini
llama-3.1-8b-instant
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
llama-3.3-70b-versatile


## Building Pipeline with both models

In [3]:
load_dotenv()

# load retriever
retriever = get_retriever(path='../')

# Model 1: current (Milestone 2)
llm_1 = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"))

# Model 2: new
llm_2 = ChatGroq(model="qwen/qwen3-32b", api_key=os.getenv("GROQ_API_KEY"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from ../data/processed/faiss_index
Loading retriever...
retriever loaded from ../data/processed/bm25_index
Creating hybrid retriever...
Completed hybrid retriever!


In [4]:
def build_chain(llm):
    return (
        {"context": retriever | build_context, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

chain_1 = build_chain(llm_1)
chain_2 = build_chain(llm_2)

### Comparing 5 queries

In [5]:
# queries = [
#     "What Nintendo Switch games are good for young children?",
#     "Comfortable gaming headsets",
#     "What are some good racing games for Xbox?",
#     "Affordable wireless controller with good battery life",
#     "What gaming accessories would improve my setup under $50?"
# ]

In [6]:
query1 = "What Nintendo Switch games are good for young children?"

#original model llama
print(f"Query: {query1}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query1))

#new model qwen
print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query1))

Query: What Nintendo Switch games are good for young children?

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the reviews and ratings provided, I recommend the following Nintendo Switch games that are suitable for young children:

1. **Smart Boys Winter Wonderland** (Rating: 4.4/5, Price: $9.94) - This game is mentioned as being great for preschoolers and is a fun, engaging experience for young children.
2. **51 Worldwide Games** (Rating: 4.6/5, Price: $35.97) - This game is a great option for young children as it contains a variety of games that can be played with family members of different ages. One reviewer mentions that their 4-year-old grandson loves to play board games with them using this console.
3. **Super Mario Maker 2** (Rating: 4.6/5, Price: $N/A) - Although the price is not listed, this game is mentioned as being a great value and is loved by grandkids aged 7 and 15. It's a fun and creat

In [7]:
query2 = "Comfortable gaming headsets"

print(f"Query: {query2}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query2))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query2))

Query: Comfortable gaming headsets

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the reviews and descriptions provided, I'd recommend the following products for comfortable gaming headsets:

1. **Fantech Gaming Headset**: With a rating of 5.0/5, this product is highly praised for its comfort. The review states, "very comfortable great sound quality." This indicates that the headset is designed for long gaming sessions without any discomfort.
2. **Rehoc 35mm Gaming Headset**: Although the price is not specified, this product has a rating of 5.0/5 and is described as "pretty and light up when plugged in" and "very comfortable." This suggests that it might be a good option for those seeking a comfortable gaming experience.
3. **Arctis 9x Replacement Earpad**: Although this is an earpad replacement, the reviews suggest that it's a good product with a high rating of 4.6/5. The review states, "great produc

In [8]:
query3 = "What are some good racing games for Xbox?"

print(f"Query: {query3}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query3))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query3))

Query: What are some good racing games for Xbox?

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the given context, I would recommend the following racing games for Xbox:

1. **Forza 3 Ultimate Platinum Hits**:
This game has a rating of 4.0/5 and is described as a realistic racing experience with over 400 cars and 100 tracks. Reviewers have praised its graphics, gameplay, and value. One reviewer mentioned that it's a "steal" at $20, which is a great price for this game.

Review Quote: "This was a steal at only 20 dollars tons of extra content and a fun base game  as realistic as they get a ton of friends when combined with a wheel controller"

Price: $20.11

2. **Tony Stewart's All-American Racing**:
This game has a rating of 4.6/5 and is described as a fun and realistic racing game on the dirt. Reviewers have praised its gameplay and graphics. One reviewer mentioned that it's a "great game" and that t

In [9]:
query4 = "Affordable wireless controller with good battery life"

print(f"Query: {query4}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query4))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query4))

Query: Affordable wireless controller with good battery life

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the context you've provided, I recommend the following affordable wireless controllers with good battery life:

1. **Feirsh Switch Controller Wireless Pro** - Rating: 4.4/5, Price: $24.99
This controller has received numerous positive reviews for its comfort, quality, and battery life. Many reviewers have mentioned that it's a great value for its price and is perfect for kids.

Review Evidence: "great controller good quality it is some light but not to much battery life is good  ergonomic valuable for money... long battery life very comfortable great quality and responsiveness..." - Reviewer, 4.4/5 stars

2. **Joyork Wireless Controller** - Rating: 3.5/5, Price: unknown
Although the rating is lower, some reviewers have mentioned that it works great and has a long battery life, which meets your r

In [10]:
query5 = "What gaming accessories would improve my setup under $50?"

print(f"Query: {query5}")
print(f"\n{'─'*50}")
print("llama-3.1-8b-instant")
print('─'*50)
print(chain_1.invoke(query5))

print(f"\n{'─'*50}")
print("qwen/qwen3-32b")
print('─'*50)
print(chain_2.invoke(query5))

Query: What gaming accessories would improve my setup under $50?

──────────────────────────────────────────────────
llama-3.1-8b-instant
──────────────────────────────────────────────────
Based on the provided context, I recommend the following gaming accessories that can improve your setup under $50:

1. **Cooper Master Dual-Sided Gaming Mouse Pad with RGB Illumination and Software Customization**: This product has a rating of 4.3/5 and a price of $17.96. The reviews praise its stylish and functional design, making it a great addition to any gaming setup. The RGB illumination and software customization features will enhance your gaming experience.

2. **Eureka Ergonomic Call of Duty PC Computer Gaming Desk Setup Accessories Bundle**: This product has a rating of 4.5/5 and a price of $29.99. The reviews highlight its sturdy and well-designed construction, making it a great investment for any gamer. The bundle includes a controller stand, USB charger, and other useful accessories that 

# Tool Integration

In [2]:
from src.hybrid_rag_pipeline import hybrid_rag_pipeline, get_retriever
retriever = get_retriever('../')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from ../data/processed/faiss_index
Loading retriever...
retriever loaded from ../data/processed/bm25_index
Creating hybrid retriever...
Completed hybrid retriever!


In [3]:
print(hybrid_rag_pipeline(retriever, 'new release single player game'))

**The Defiant**
- **Price:** $59.99
- **Rating:** 4.7/5
- **Release year:** 2026
- **Why it matches:** The Defiant is a new release single player game that offers a gripping story and challenging gameplay.

**STALKER 2 Cost of Hope**
- **Price:** $59.99
- **Rating:** 4.5/5
- **Release year:** 2026
- **Why it matches:** STALKER 2 Cost of Hope is a new release single player game that offers a deep experience with a personalized story and challenging gameplay.

**Alan Wake 2**
- **Price:** $59.99
- **Rating:** 4.5/5
- **Release year:** 2025
- **Why it matches:** Alan Wake 2 is a new release single player game that offers a gripping story and devastatingly effective art style, making it one of the best single-player games of recent years.


In [4]:
print(hybrid_rag_pipeline(retriever, 'pink wireless controller'))

**Xbox 360 Wireless Controller - Pink**
- **Price:** $42.87
- **Rating:** 4.0/5
- **Release year:** 2007
- **Why it matches:** The product matches the user's query because it is a pink wireless controller for the Xbox 360, which is a type of gaming console that the user may be interested in. The product's price is also listed as $42.87, and it has a rating of 4.0 out of 5 stars.


In [5]:
print(hybrid_rag_pipeline(retriever, 'recent single player games under 20$'))

**Expedition 33**
- **Price:** $19.99
- **Rating:** 4.5/5
- **Release year:** (2022)
- **Why it matches:** A single-player focused survival game with a strong narrative and engaging gameplay.

**Bloodborne**
- **Price:** $19.99
- **Rating:** 5/5
- **Release year:** (2015)
- **Why it matches:** A challenging action RPG with a dark atmosphere and intense combat.

**RE4 remake**
- **Price:** $19.99
- **Rating:** 4.5/5
- **Release year:** (2023)
- **Why it matches:** A re-imagining of the classic Resident Evil 4 game with updated graphics and gameplay.

**Sekiro (Note: This game is not under 20$ but I have to recommend it)**
- **Price:** $59.99
- **Rating:** 4.5/5
- **Release year:** (2019)
- **Why it matches:** A challenging action-adventure game set in feudal Japan with a strong focus on stealth and strategy.
